# 🏥 Disease Prediction & AI Chatbot — All-in-One Notebook

This single notebook contains **everything**: model loading, FastAPI backend, and a built-in web UI (upload button, predict button, and chatbot) — served directly by FastAPI. No separate frontend files, no npm, no React, no Docker.

**How to use:**
1. Put your trained `.pth` file in the same folder as this notebook (or update `MODEL_PATH` below).
2. Run every cell top to bottom (Cell → Run All).
3. Edit the **Configuration** cell with your Google API key, model architecture, and class names.
4. Once the last cell runs, open **http://localhost:8000** in your browser.
5. To stop the server, restart the notebook kernel (Kernel → Restart).

⚠️ This assumes you already trained your model separately — this notebook only loads it for inference and serving.

## 1. Install Dependencies
Run once. If already installed, this will just confirm versions.

In [ ]:
%pip install -q fastapi uvicorn python-multipart pillow torch torchvision numpy google-generativeai python-dotenv
print("Dependencies installed.")


Dependencies installed.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

In [2]:
import os
import io
import json
import threading
import time
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv

import numpy as np
import torch
import torchvision
from PIL import Image

from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
from fastapi.concurrency import run_in_threadpool
from pydantic import BaseModel
import uvicorn

import google.generativeai as genai

print("Imports OK. PyTorch version:", torch.__version__)


Imports OK. PyTorch version: 2.13.0+cpu


d:\Academic_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ankan\AppData\Local\Temp\ipykernel_2508\1512733567.py:23: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 3. Configuration — ⚠️ EDIT THESE VALUES

In [3]:
# ============ CONFIGURATION - EDIT THESE VALUES ============

# Loads variables from a local .env file (NOT committed to Git — see .gitignore)
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")   # Reads from .env — never type your key directly here
if not GOOGLE_API_KEY:
    print("⚠️ GOOGLE_API_KEY not found. Create a .env file next to this notebook with:")
    print('   GOOGLE_API_KEY=your_actual_key_here')

MODEL_PATH = "fine_tuned_skin_disease_model_unfrozen.pth"   # Place this file next to the notebook
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_ARCHITECTURE = "resnet18"                # Detected from your .pth: BasicBlock, 2 blocks/layer = ResNet18
NUM_CLASSES = 8                                 # Detected from your .pth: fc.weight shape = [8, 512]
CLASS_NAMES = [
    "BA-cellulitis",
    "BA-impetigo",
    "FU-athlete-foot",
    "FU-nail-fungus",
    "FU-ringworm",
    "PA-cutaneous-larva-migrans",
    "VI-chickenpox",
    "VI-shingles",
]                                                # ⚠️ REQUIRED: your .pth stores no class names.
                                                 # Replace these with your real 8 skin disease labels,
                                                 # in the EXACT order used during training.
                                                 # If you used torchvision.datasets.ImageFolder, that
                                                 # order is alphabetical by folder name — check your
                                                 # training notebook/script for train_dataset.classes.

IMAGE_SIZE = (224, 224)                         # Standard ResNet input — change if your training used a different size
IMAGE_MEAN = [0.485, 0.456, 0.406]              # Update if you used custom normalization during training
IMAGE_STD = [0.229, 0.224, 0.225]

MAX_FILE_SIZE = 10 * 1024 * 1024                # 10MB
SERVER_PORT = 8000

print("Configuration loaded.")
print(f"Device: {DEVICE}")
print(f"Model path: {MODEL_PATH}  (exists: {Path(MODEL_PATH).exists()})")
print(f"Google API key loaded: {'Yes' if GOOGLE_API_KEY else 'No'}")


Configuration loaded.
Device: cpu
Model path: fine_tuned_skin_disease_model_unfrozen.pth  (exists: True)
Google API key loaded: Yes


## 4. Model Definition & Loading
If your training file used a **different architecture** than the ones below (resnet50/resnet101/vgg16/efficientnet_b0), add an `elif` branch in `_build_model` that matches your exact training code — the architecture here must be identical to what you trained, only the final layer changes for `NUM_CLASSES`.

In [4]:
class DiseasePredictor:
    def __init__(self, model_path, device, architecture="resnet50", num_classes=10, class_names=None):
        self.device = device
        self.class_names = class_names
        self.model = self._build_model(architecture, num_classes)
        self._load_weights(model_path)
        self.model.to(self.device)
        self.model.eval()

    def _build_model(self, architecture, num_classes):
        if architecture == "resnet18":
            model = torchvision.models.resnet18(weights=None)
            model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
        elif architecture == "resnet34":
            model = torchvision.models.resnet34(weights=None)
            model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
        elif architecture == "resnet50":
            model = torchvision.models.resnet50(weights=None)
            model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
        elif architecture == "resnet101":
            model = torchvision.models.resnet101(weights=None)
            model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
        elif architecture == "vgg16":
            model = torchvision.models.vgg16(weights=None)
            model.classifier[-1] = torch.nn.Linear(model.classifier[-1].in_features, num_classes)
        elif architecture == "efficientnet_b0":
            model = torchvision.models.efficientnet_b0(weights=None)
            model.classifier[-1] = torch.nn.Linear(model.classifier[-1].in_features, num_classes)
        else:
            raise ValueError(
                f"Unsupported architecture: {architecture}. "
                "Add a matching branch here that mirrors your training script."
            )
        return model

    def _load_weights(self, model_path):
        if not Path(model_path).exists():
            raise FileNotFoundError(f"Model file not found: {model_path}")
        checkpoint = torch.load(model_path, map_location=self.device)

        if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
            if checkpoint.get("class_names"):
                self.class_names = checkpoint["class_names"]
        elif isinstance(checkpoint, dict) and "model" in checkpoint:
            state_dict = checkpoint["model"]
        else:
            state_dict = checkpoint

        self.model.load_state_dict(state_dict)
        print(f"✅ Model weights loaded from {model_path}")

    def preprocess(self, image: Image.Image) -> torch.Tensor:
        image = image.convert("RGB").resize(IMAGE_SIZE)
        arr = np.array(image).astype(np.float32) / 255.0
        arr = (arr - np.array(IMAGE_MEAN)) / np.array(IMAGE_STD)
        tensor = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).float()
        return tensor.to(self.device)

    def predict(self, image: Image.Image) -> dict:
        tensor = self.preprocess(image)
        with torch.no_grad():
            outputs = self.model(tensor)
            probs = torch.nn.functional.softmax(outputs, dim=1)
            confidence, idx = torch.max(probs, 1)
        idx = idx.item()
        confidence = confidence.item()
        if self.class_names and idx < len(self.class_names):
            name = self.class_names[idx]
        else:
            name = f"Class_{idx}"
        return {"disease": name, "confidence": round(confidence * 100, 2), "class_index": idx}


In [5]:
try:
    predictor = DiseasePredictor(
        model_path=MODEL_PATH,
        device=DEVICE,
        architecture=MODEL_ARCHITECTURE,
        num_classes=NUM_CLASSES,
        class_names=CLASS_NAMES,
    )
    print("✅ Model ready for inference")
except Exception as e:
    predictor = None
    print(f"❌ Failed to load model: {e}")
    print("Fix MODEL_PATH / MODEL_ARCHITECTURE / NUM_CLASSES above and re-run this cell.")


✅ Model weights loaded from fine_tuned_skin_disease_model_unfrozen.pth
✅ Model ready for inference


## 5. Configure Google Generative AI (Chatbot)

In [6]:
chat_model = None
vision_model = None

if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)

    try:
        available = [
            m.name for m in genai.list_models()
            if "generateContent" in m.supported_generation_methods
        ]
        print("Models available to your API key:")
        for name in available:
            print(" -", name)

        # Preferred fast/current multimodal models, checked in order
        preferred_order = [
            "models/gemini-flash-latest",
            "models/gemini-2.0-flash",
            "models/gemini-1.5-flash-latest",
            "models/gemini-1.5-flash",
            "models/gemini-pro-latest",
        ]
        selected = next((m for m in preferred_order if m in available), None)
        if selected is None and available:
            selected = available[0]   # fall back to whatever IS available

        if selected:
            chat_model = genai.GenerativeModel(selected)
            vision_model = chat_model   # same model handles both text and images
            print(f"\n✅ Using model: {selected}")
        else:
            print("\n❌ No models supporting generateContent are available to this API key")

    except Exception as e:
        print(f"❌ Could not list models: {e}")
        print("Double-check GOOGLE_API_KEY is valid and has Generative Language API access")
else:
    print("⚠️ GOOGLE_API_KEY not set — chatbot and AI analysis will be disabled")
    print("Add it to your .env file and re-run this notebook")


Models available to your API key:
 - models/gemini-2.5-flash
 - models/gemini-2.5-pro
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-2.5-pro-preview-tts
 - models/gemma-4-26b-a4b-it
 - models/gemma-4-31b-it
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-pro-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-flash-preview
 - models/gemini-3.1-pro-preview
 - models/gemini-3.1-pro-preview-customtools
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3.1-flash-lite
 - models/gemini-3-pro-image-preview
 - models/gemini-3-pro-image
 - models/nano-banana-pro-preview
 - models/gemini-3.1-flash-image-preview
 - models/gemini-3.1-flash-image
 - models/gemini-3.1-flash-lite-image
 - models/gemini-3.5-flash
 - models/gemini-3.5-flash-lite
 - models/gemini-omni-flash-preview
 - models/gemini-3.6-flash
 - models/gemini-3.7-flash
 - models/lyria-3-clip-preview
 - models/lyria-3-pro-preview
 - models/gemini-3.1-

## 6. FastAPI App

In [7]:
app = FastAPI(title="Disease Prediction & AI Chatbot")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

chat_history = []

class ChatMessage(BaseModel):
    message: str


## 7. Built-in Web UI
This is the entire frontend (HTML + CSS + JS) as one string. FastAPI serves it directly at `/` — no React, no npm, no build step.

In [8]:
HTML_PAGE = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Disease Prediction & AI Assistant</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: "Segoe UI", Tahoma, Geneva, Verdana, sans-serif; background: linear-gradient(135deg,#667eea,#764ba2); min-height: 100vh; padding: 20px; }
  .app { max-width: 900px; margin: 0 auto; background: #fff; border-radius: 20px; box-shadow: 0 20px 60px rgba(0,0,0,.3); overflow: hidden; }
  header { background: linear-gradient(135deg,#667eea,#764ba2); color: #fff; padding: 30px; display:flex; justify-content:space-between; align-items:center; flex-wrap: wrap; gap: 10px;}
  header h1 { font-size: 1.7em; }
  header p { opacity:.9; font-size:.95em; }
  #status { padding:6px 14px; border-radius:20px; font-weight:600; font-size:.85em; background:#f44336; }
  #status.ok { background:#4caf50; }
  .tabs { display:flex; gap:10px; padding:20px 30px 0; border-bottom:2px solid #f0f0f0; }
  .tab-btn { padding:10px 20px; border:none; background:none; font-weight:600; color:#666; cursor:pointer; border-bottom:3px solid transparent; }
  .tab-btn.active { color:#667eea; border-bottom-color:#667eea; }
  main { padding: 30px; }
  .tabpanel { display:none; }
  .tabpanel.active { display:block; }
  .dropzone { border:3px dashed #667eea; border-radius:12px; padding:40px; text-align:center; cursor:pointer; background:#f5f5f5; margin-bottom:20px; }
  .dropzone:hover { background:#efefff; }
  #preview { max-width:100%; border-radius:10px; margin-bottom:15px; display:none; }
  .btnrow { display:flex; gap:12px; margin-bottom:20px; flex-wrap: wrap;}
  button.action { padding:12px 26px; border:none; border-radius:10px; font-weight:600; cursor:pointer; }
  #predictBtn { background:linear-gradient(135deg,#667eea,#764ba2); color:#fff; }
  #predictBtn:disabled { opacity:.5; cursor:not-allowed; }
  #resetBtn { background:#eee; color:#333; }
  .card { background:#f9f9f9; border:2px solid #e0e0e0; border-radius:14px; padding:20px; margin-bottom:16px; }
  .disease-name { font-size:1.4em; font-weight:700; color:#667eea; margin:8px 0; }
  .confbar { height:10px; background:#e0e0e0; border-radius:5px; overflow:hidden; margin:6px 0; }
  .confbar-fill { height:100%; background:linear-gradient(90deg,#4caf50,#8bc34a); }
  .analysis { white-space:pre-wrap; line-height:1.6; color:#444; }
  .disclaimer { background:#e3f2fd; border-left:4px solid #2196f3; padding:12px 16px; border-radius:8px; color:#1565c0; font-size:.9em; }
  .error { background:#ffebee; color:#c62828; border-left:4px solid #c62828; padding:12px 16px; border-radius:8px; margin-bottom:16px; }
  #chatBox { height:380px; overflow-y:auto; border:2px solid #e0e0e0; border-radius:12px; padding:15px; background:#f9f9f9; display:flex; flex-direction:column; gap:10px; margin-bottom:15px; }
  .msg { max-width:75%; padding:10px 14px; border-radius:10px; }
  .msg.user { align-self:flex-end; background:linear-gradient(135deg,#667eea,#764ba2); color:#fff; }
  .msg.assistant { align-self:flex-start; background:#eee; color:#333; }
  .chatform { display:flex; gap:10px; }
  #chatInput { flex:1; padding:12px; border:2px solid #e0e0e0; border-radius:10px; font-size:1em; }
  #sendBtn { background:linear-gradient(135deg,#667eea,#764ba2); color:#fff; }
  small.hint { color:#999; display:block; margin-top:6px; }
</style>
</head>
<body>
<div class="app">
  <header>
    <div>
      <h1>🏥 Disease Prediction & AI Assistant</h1>
      <p>Upload an image, get a prediction, ask the assistant</p>
    </div>
    <div id="status">Checking...</div>
  </header>

  <div class="tabs">
    <button class="tab-btn active" data-tab="predict">🔍 Predict Disease</button>
    <button class="tab-btn" data-tab="chat">💬 AI Assistant</button>
  </div>

  <main>
    <section id="predict" class="tabpanel active">
      <div id="predictError"></div>
      <div class="dropzone" id="dropzone">
        <p>📸 Click to upload or drag & drop</p>
        <small class="hint">JPG, PNG, GIF — max 10MB</small>
        <input type="file" id="fileInput" accept="image/*" style="display:none">
      </div>
      <img id="preview">
      <div class="btnrow">
        <button class="action" id="predictBtn" disabled>🚀 Predict Disease</button>
        <button class="action" id="resetBtn">🔄 Reset</button>
      </div>
      <div id="results"></div>
    </section>

    <section id="chat" class="tabpanel">
      <div id="chatBox">
        <div style="text-align:center;color:#999;">👋 Ask me about diseases, symptoms, or general health advice.</div>
      </div>
      <form class="chatform" id="chatForm">
        <input type="text" id="chatInput" placeholder="Ask a question...">
        <button type="submit" class="action" id="sendBtn">Send</button>
      </form>
    </section>
  </main>
</div>

<script>
let uploadedFile = null;

document.querySelectorAll(".tab-btn").forEach(btn => {
  btn.addEventListener("click", () => {
    document.querySelectorAll(".tab-btn").forEach(b => b.classList.remove("active"));
    document.querySelectorAll(".tabpanel").forEach(p => p.classList.remove("active"));
    btn.classList.add("active");
    document.getElementById(btn.dataset.tab).classList.add("active");
  });
});

async function checkHealth() {
    try {
        const res = await fetch("/health");
        const data = await res.json();

        const el = document.getElementById("status");

        if (data.model_loaded && data.api_configured) {
            el.textContent = "✓ Model & AI Ready";
            el.classList.add("ok");
        }
        else if (data.model_loaded && !data.api_configured) {
            el.textContent = "⚠ Model Ready • AI Offline";
            el.classList.remove("ok");
        }
        else {
            el.textContent = "✗ Model Not Loaded";
            el.classList.remove("ok");
        }

    } catch (e) {
        document.getElementById("status").textContent =
            "✗ Server Error";
    }
}
checkHealth();

const dropzone = document.getElementById("dropzone");
const fileInput = document.getElementById("fileInput");
const preview = document.getElementById("preview");
const predictBtn = document.getElementById("predictBtn");
const resetBtn = document.getElementById("resetBtn");
const resultsDiv = document.getElementById("results");
const predictError = document.getElementById("predictError");

dropzone.addEventListener("click", () => fileInput.click());
dropzone.addEventListener("dragover", e => { e.preventDefault(); dropzone.style.borderColor = "#764ba2"; });
dropzone.addEventListener("dragleave", () => { dropzone.style.borderColor = "#667eea"; });
dropzone.addEventListener("drop", e => {
  e.preventDefault();
  dropzone.style.borderColor = "#667eea";
  if (e.dataTransfer.files.length) handleFile(e.dataTransfer.files[0]);
});
fileInput.addEventListener("change", e => { if (e.target.files.length) handleFile(e.target.files[0]); });

function handleFile(file) {
  predictError.innerHTML = "";
  if (!["image/jpeg","image/png","image/gif"].includes(file.type)) {
    predictError.innerHTML = '<div class="error">⚠️ Please upload a JPG, PNG, or GIF file</div>';
    return;
  }
  if (file.size > 10 * 1024 * 1024) {
    predictError.innerHTML = '<div class="error">⚠️ File exceeds 10MB limit</div>';
    return;
  }
  uploadedFile = file;
  const reader = new FileReader();
  reader.onload = e => { preview.src = e.target.result; preview.style.display = "block"; };
  reader.readAsDataURL(file);
  predictBtn.disabled = false;
  resultsDiv.innerHTML = "";
}

resetBtn.addEventListener("click", () => {
  uploadedFile = null;
  fileInput.value = "";
  preview.style.display = "none";
  predictBtn.disabled = true;
  resultsDiv.innerHTML = "";
  predictError.innerHTML = "";
});

predictBtn.addEventListener("click", async () => {
    if (!uploadedFile) return;

    predictBtn.disabled = true;
    predictBtn.textContent = "⏳ Predicting...";
    predictError.innerHTML = "";
    resultsDiv.innerHTML = "";

    try {
        // ==========================================
        // 1. FAST CNN PREDICTION
        // ==========================================

        const predictionForm = new FormData();
        predictionForm.append("file", uploadedFile);

        const predictionResponse = await fetch("/predict", {
            method: "POST",
            body: predictionForm
        });

        if (!predictionResponse.ok) {
            const errorText = await predictionResponse.text();

            throw new Error(
                `Prediction failed (${predictionResponse.status}): ${errorText}`
            );
        }

        const predictionData = await predictionResponse.json();

        console.log("Prediction:", predictionData);

        const disease = predictionData.disease;
        const confidence = Number(predictionData.confidence);

        // ==========================================
        // 2. SHOW CNN RESULT IMMEDIATELY
        // ==========================================

        resultsDiv.innerHTML = `
            <div class="card">
                <h3>🔬 Prediction Result</h3>

                <div class="disease-name">
                    ${disease}
                </div>

                <div class="confbar">
                    <div
                        class="confbar-fill"
                        style="width:${Math.min(Math.max(confidence, 0), 100)}%"
                    ></div>
                </div>

                <div>
                    ${confidence.toFixed(2)}% model confidence
                </div>
            </div>

            <div class="card">
                <h3>🤖 AI Analysis</h3>

                <div class="analysis" id="aiAnalysis">
                    ⏳ Generating AI analysis...
                </div>
            </div>

            <div class="disclaimer">
                <strong>⚕️ Medical Disclaimer:</strong>
                This is an AI-generated classification result for
                informational purposes only. It is not a medical diagnosis.
                Please consult a qualified healthcare professional.
            </div>
        `;

        predictBtn.textContent = "✓ Prediction Complete";

        // ==========================================
        // 3. SEND AI ANALYSIS REQUEST
        // ==========================================

        const analysisForm = new FormData();

        analysisForm.append("file", uploadedFile);
        analysisForm.append("disease", disease);
        analysisForm.append("confidence", confidence.toString());

        console.log("Sending /analysis request...");

        // Do NOT await this.
        // Gemini can work while the prediction remains visible.
        fetch("/analysis", {
            method: "POST",
            body: analysisForm
        })
        .then(async response => {

            if (!response.ok) {
                const errorText = await response.text();

                throw new Error(
                    `Analysis failed (${response.status}): ${errorText}`
                );
            }

            return response.json();
        })
        .then(data => {

            console.log("AI Analysis:", data);

            const aiAnalysis =
                document.getElementById("aiAnalysis");

            if (aiAnalysis) {
                aiAnalysis.textContent =
                    data.ai_analysis ||
                    "AI analysis could not generate a response.";
            }
        })
        .catch(error => {

            console.error("AI Analysis Error:", error);

            const aiAnalysis =
                document.getElementById("aiAnalysis");

            if (aiAnalysis) {
                aiAnalysis.textContent =
                    "⚠️ AI analysis unavailable: " +
                    error.message;
            }
        });

    } catch (err) {

        console.error("Prediction Error:", err);

        predictError.innerHTML =
            `<div class="error">⚠️ ${err.message}</div>`;

        predictBtn.textContent = "🚀 Predict Disease";
    }

    predictBtn.disabled = false;
});

const chatBox = document.getElementById("chatBox");
const chatForm = document.getElementById("chatForm");
const chatInput = document.getElementById("chatInput");

function addMessage(role, text) {
  const div = document.createElement("div");
  div.className = "msg " + role;
  div.textContent = (role === "user" ? "👤 " : "🤖 ") + text;
  chatBox.appendChild(div);
  chatBox.scrollTop = chatBox.scrollHeight;
}

chatForm.addEventListener("submit", async e => {
  e.preventDefault();
  const message = chatInput.value.trim();
  if (!message) return;
  addMessage("user", message);
  chatInput.value = "";

  try {
    const res = await fetch("/chat", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ message })
    });
    const data = await res.json();
    if (!res.ok) throw new Error(data.detail || "Chat failed");
    addMessage("assistant", data.response);
  } catch (err) {
    addMessage("assistant", "Sorry, I ran into an error: " + err.message);
  }
});
</script>
</body>
</html>
"""
print("HTML frontend defined (" + str(len(HTML_PAGE)) + " characters)")


HTML frontend defined (13199 characters)


## 8. API Routes

In [9]:
@app.get("/", response_class=HTMLResponse)
async def root():
    return HTML_PAGE

@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "model_loaded": predictor is not None,
        "api_configured": chat_model is not None,
        "vision_configured": vision_model is not None,
    }

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    if predictor is None:
        raise HTTPException(status_code=500, detail="Model not loaded")

    if file.content_type not in ["image/jpeg", "image/png", "image/gif"]:
        raise HTTPException(
            status_code=400,
            detail="Only JPEG, PNG, GIF files are allowed"
        )

    contents = await file.read()

    if len(contents) > MAX_FILE_SIZE:
        raise HTTPException(
            status_code=413,
            detail="File too large (max 10MB)"
        )

    try:
        image = Image.open(io.BytesIO(contents)).convert("RGB")

        # Run blocking CNN inference outside FastAPI's event loop
        prediction = await run_in_threadpool(
            predictor.predict,
            image
        )

        return {
            **prediction,
            "timestamp": datetime.now().isoformat()
        }

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Prediction failed: {e}"
        )

@app.post("/analysis")
async def analysis(
    file: UploadFile = File(...),
    disease: str = Form(...),
    confidence: float = Form(...)
):
    if vision_model is None:
        raise HTTPException(
            status_code=503,
            detail="AI analysis unavailable — set GOOGLE_API_KEY"
        )

    if file.content_type not in [
        "image/jpeg",
        "image/png",
        "image/gif"
    ]:
        raise HTTPException(
            status_code=400,
            detail="Only JPEG, PNG, GIF files are allowed"
        )

    contents = await file.read()

    if len(contents) > MAX_FILE_SIZE:
        raise HTTPException(
            status_code=413,
            detail="File too large (max 10MB)"
        )

    try:
        image = Image.open(
            io.BytesIO(contents)
        ).convert("RGB")

        prompt = (
            f"A skin-disease classification model predicted "
            f"the class '{disease}' with a model confidence score "
            f"of {confidence:.2f}%.\n\n"

            "Provide a concise educational explanation based on "
            "this prediction.\n\n"

            "Do NOT present this prediction as a confirmed "
            "medical diagnosis.\n"

            "Do NOT interpret the confidence score as the "
            "probability that the person actually has the condition.\n\n"

            "Include exactly these sections:\n"
            "1. Possible condition\n"
            "2. Common symptoms\n"
            "3. When to seek medical attention\n"
            "4. General prevention\n\n"

            "Keep the response concise and easy to understand. "
            "Clearly state that image-based AI predictions can be "
            "incorrect and that a qualified dermatologist or "
            "healthcare professional should evaluate the lesion."
        )

        response = await run_in_threadpool(
            vision_model.generate_content,
            [prompt, image]
        )

        if response and hasattr(response, "text") and response.text:
            analysis_text = response.text
        else:
            analysis_text = (
                "AI analysis could not generate a valid response."
            )

        return {
            "ai_analysis": analysis_text
        }

    except Exception as e:
        print("ANALYSIS ERROR:", repr(e))

        raise HTTPException(
            status_code=500,
            detail=f"AI analysis failed: {str(e)}"
        )
        
@app.post("/chat")
async def chat(payload: ChatMessage):

    if chat_model is None:
        raise HTTPException(
            status_code=503,
            detail="Chat AI is not configured. Check GOOGLE_API_KEY and chat_model initialization."
        )

    message = payload.message.strip()

    if not message:
        raise HTTPException(
            status_code=400,
            detail="Message cannot be empty"
        )

    chat_history.append({
        "role": "user",
        "content": message
    })

    system_context = (
        "You are a helpful medical information assistant. "
        "Provide general health information and always remind users "
        "to consult healthcare professionals for diagnosis and treatment. "
        "Be empathetic and concise."
    )

    try:

        prompt = (
            f"{system_context}\n\n"
            f"User: {message}"
        )

        response = await run_in_threadpool(
            chat_model.generate_content,
            prompt
        )

        if not response:
            raise Exception("Empty response from AI model")

        reply = getattr(response, "text", None)

        if not reply:
            raise Exception("AI returned no text response")

    except Exception as e:

        print("CHAT ERROR:", repr(e))

        # Remove the user's message if AI failed
        if chat_history and chat_history[-1]["role"] == "user":
            chat_history.pop()

        raise HTTPException(
            status_code=500,
            detail=f"AI chat failed: {str(e)}"
        )

    chat_history.append({
        "role": "assistant",
        "content": reply
    })

    if len(chat_history) > 20:
        del chat_history[:-20]

    return {
        "response": reply,
        "role": "assistant"
    }
    
print("\n========== ROUTES ==========")

for route in app.routes:
    print(route.methods, route.path)

print("============================\n")


========== ROUTES ==========
{'HEAD', 'GET'} /openapi.json
{'HEAD', 'GET'} /docs
{'HEAD', 'GET'} /docs/oauth2-redirect
{'HEAD', 'GET'} /redoc
{'GET'} /
{'GET'} /health
{'POST'} /predict
{'POST'} /analysis
{'POST'} /chat



## 9. Run the Server
This starts FastAPI in a background thread so the notebook stays usable. Once you see the ready message, open **http://localhost:8000**.

In [ ]:
def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=SERVER_PORT, log_level="info")

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()
time.sleep(2)

print(f"🚀 Server running at http://localhost:{SERVER_PORT}")
print("Open that URL in your browser to use the app.")
print("To stop the server: Kernel → Restart (or Interrupt) this notebook.")


INFO:     Started server process [2508]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 Server running at http://localhost:8000
Open that URL in your browser to use the app.
To stop the server: Kernel → Restart (or Interrupt) this notebook.


INFO:     127.0.0.1:61910 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:61910 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:63783 - "POST /predict HTTP/1.1" 200 OK
ANALYSIS ERROR: ResourceExhausted('You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.7-flash\nPlease retry in 22.133521947s.')
INFO:     127.0.0.1:63783 - "POST /analysis HTTP/1.1" 500 Internal Server Error
INFO:     127.0.0.1:58101 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:58101 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:58084 - "POST /predict HTTP/1.1" 200 OK
ANALYSIS ERROR: ResourceExhausted('You exceeded your current quota, please check your plan and billing details. For more information o

## Notes

- **Stopping the server:** Since it runs in a background thread, use *Kernel → Restart* to fully stop it. Re-run the cells to start again.
- **Changing the model:** If you retrain and save a new `.pth`, just update `MODEL_PATH` in the Configuration cell and re-run from cell 4 (Model Definition) onward — no need to restart the server cell unless you also change the architecture.
- **Architecture mismatch errors** (`size mismatch` / `Missing key(s)`) usually mean `MODEL_ARCHITECTURE` or `NUM_CLASSES` here don't exactly match your training script. Copy the exact model-building code from your training file into `_build_model()`.
- **No Google API key?** The predict button and disease classification will still work — only the AI-written analysis and chatbot will be disabled until you add a key.
- **Testing the API directly:** FastAPI's auto docs are available at http://localhost:8000/docs while the server is running.